# ToysAD-8K Post-freeze Full-pool Frozen Inference

**Notebook file:** `11a-toysad8k-full-pool-frozen-inference.ipynb`

This optional descriptive sensitivity notebook applies the exact frozen LocateAnything, SAM 2.1, DINOv2-small, identity and P1 route to the 4,107-scene eligible training pool. It runs only after the official result is complete and Notebook 11R has released a separate model-facing resource.

The run is prediction-only. It never mounts evaluator labels or geometry and cannot change the method, prompt, view contracts, fusion rule or review policy.

## Inference and recovery contract

- A bounded one-image preflight loads LocateAnything, SAM 2.1 and DINOv2-small in sequence before a long run starts.
- `gate` then processes one shard through every component before a longer run.
- `incremental` processes a bounded number of new shards and exports a hash-locked recovery bundle.
- `full` publishes immutable predictions only after all 82,140 images have a result or a retained failure.
- Exact stage receipts prevent reuse after any input, method or policy change.
- The saved evidence includes runtime, terminal failures, view-contract tables, review signals and prediction-only panels with boxes, masks, masked objects, identities, ranks and fused odd-one-out results.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True
NOTEBOOK_FILE = "11a-toysad8k-full-pool-frozen-inference.ipynb"
RUN_MODE = os.getenv("VPI_RUN_MODE", "gate").strip().lower()
if RUN_MODE not in {"gate", "incremental", "full"}:
    raise ValueError("VPI_RUN_MODE must be gate, incremental or full")
OUTPUT_ROOT = Path(os.getenv("VPI_OUTPUT_ROOT", "/kaggle/working/toysad8k_full_pool_inference")).expanduser().resolve()
RECOVERY_ROOT = Path(os.getenv("VPI_RECOVERY_OUTPUT", "/kaggle/working/toysad8k_full_pool_inference_recovery")).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RECOVERY_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def kaggle_roots() -> list[Path]:
    root = Path("/kaggle/input")
    return sorted((path for path in root.iterdir() if path.is_dir()), key=lambda path: path.as_posix()) if root.is_dir() else []


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        for resolved in (candidate, candidate / "code"):
            if (resolved / "scripts/run_full_pool_release.py").is_file():
                return resolved.resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def find_named_file(name: str, variable: str) -> Path:
    configured = os.getenv(variable)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    matches = sorted({path.resolve() for root in kaggle_roots() for path in root.rglob(name) if path.is_file()})
    if not matches:
        raise FileNotFoundError(f"Could not locate {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in matches}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} differ")
    return matches[0]


def find_root_with(marker: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend(kaggle_roots())
    matches = []
    for root in candidates:
        if (root / marker).is_file():
            matches.append(root.resolve())
        matches.extend(path.parent.resolve() for path in root.rglob(marker) if path.is_file())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise ValueError(f"Expected one mounted root containing {marker}, found {len(unique)}")
    return unique[0]


def find_model_input(cohort_id: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else kaggle_roots()
    matches = []
    for root in candidates:
        records = [root / "package_record.json"] if (root / "package_record.json").is_file() else []
        records.extend(path for path in root.rglob("package_record.json") if path.is_file())
        for record in records:
            try:
                value = json.loads(record.read_text(encoding="utf-8"))
            except Exception:
                continue
            if value.get("cohort_id") == cohort_id:
                matches.append(record.parent.resolve())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise ValueError(f"Expected one model package for {cohort_id}, found {len(unique)}")
    return unique[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source = str(CODE_ROOT / "src")
    existing = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source if not existing else source + os.pathsep + existing
    return environment


def clearml_mode(environment: dict[str, str]) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        access = secrets.get_secret("CLEARML_API_ACCESS_KEY")
        secret = secrets.get_secret("CLEARML_API_SECRET_KEY")
    except Exception:
        access = os.getenv("CLEARML_API_ACCESS_KEY")
        secret = os.getenv("CLEARML_API_SECRET_KEY")
    if not access or not secret:
        return "disabled"
    environment.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True,
            timeout=180,
            env=environment,
        )
    except Exception as error:
        print(f"ClearML setup was unavailable: {type(error).__name__}")
        return "disabled"
    return "online"

## Verify the frozen resources

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/post-freeze-sensitivity/11a-toysad8k-full-pool-frozen-inference.ipynb",
)
MODEL_INPUT = find_model_input("toysad8k-eligible-train-full-pool-4107-v1", "VPI_MODEL_INPUT_ROOT")
RELEASE_RECEIPT = find_named_file("full_pool_release_receipt.json", "VPI_RELEASE_RECEIPT")
METHOD_FREEZE = find_named_file("method_freeze.json", "VPI_METHOD_FREEZE")
APPLICATION_CONTRACT = find_named_file("application_output_contract.json", "VPI_APPLICATION_CONTRACT")
POLICY_FREEZE = find_named_file("review_policy_freeze.json", "VPI_POLICY_FREEZE")
SELECTED_PROMPT = find_named_file("selected_prompt.json", "VPI_SELECTED_PROMPT")
SAM_CHECKPOINT = find_named_file("sam2.1_hiera_small.pt", "VPI_SAM_CHECKPOINT")

package = json.loads((MODEL_INPUT / "package_record.json").read_text(encoding="utf-8"))
if package.get("cohort_id") != "toysad8k-eligible-train-full-pool-4107-v1":
    raise ValueError("Attached model package is not the eligible full-pool release")
display({"scene count": package["scene_count"], "image count": package["image_count"], "run mode": RUN_MODE})

## Restore one exact prediction checkpoint

In [ ]:
resume_root = os.getenv("VPI_RESUME_ROOT")
recovery_arguments = []
if resume_root:
    source = Path(resume_root).expanduser().resolve(strict=True)
    bundles = sorted(source.rglob("full-pool-inference-recovery.zip"))
    manifests = sorted(source.rglob("full-pool-inference-recovery-manifest.json"))
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("VPI_RESUME_ROOT must contain exactly one inference recovery pair")
    recovery_arguments = ["--recovery-bundle", str(bundles[0]), "--recovery-manifest", str(manifests[0])]
display({"previous recovery attached": bool(recovery_arguments)})

## Run the frozen automatic route

In [ ]:
environment = code_environment()
common_arguments = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_full_pool_inference.py"),
    "--model-input-root", str(MODEL_INPUT),
    "--release-receipt", str(RELEASE_RECEIPT),
    "--method-freeze", str(METHOD_FREEZE),
    "--application-contract", str(APPLICATION_CONTRACT),
    "--policy-freeze", str(POLICY_FREEZE),
    "--selected-prompt", str(SELECTED_PROMPT),
    "--sam-checkpoint", str(SAM_CHECKPOINT),
    "--output-root", str(OUTPUT_ROOT),
    *sum((["--mounted-root", str(root)] for root in kaggle_roots()), []),
]
preflight_path = OUTPUT_ROOT / "preflight/full-pool-model-preflight.json"
preflight_command = [
    *common_arguments,
    "--run-mode", "gate",
    "--preflight-only",
    "--preflight-output", str(preflight_path),
]
subprocess.run(preflight_command, check=True, timeout=1800, env=environment)
preflight_record = json.loads(preflight_path.read_text(encoding="utf-8"))
if preflight_record.get("status") != "passed":
    raise RuntimeError("The bounded model preflight did not pass")

tracking_mode = clearml_mode(environment)
command = [
    *common_arguments,
    "--run-mode", RUN_MODE,
    "--shard-size", os.getenv("VPI_SHARD_SIZE", "100"),
    "--maximum-new-shards", os.getenv("VPI_MAXIMUM_NEW_SHARDS", "10"),
    "--recovery-output-directory", str(RECOVERY_ROOT),
    "--clearml-mode", tracking_mode,
    *recovery_arguments,
]
if os.getenv("VPI_PRUNE_AFTER_RECOVERY", "1") == "1" and RUN_MODE != "full":
    command.append("--prune-after-recovery")
environment["HF_HUB_OFFLINE"] = "1"
environment["TRANSFORMERS_OFFLINE"] = "1"
subprocess.run(command, check=True, env=environment)
display({"model preflight": preflight_record["status"], "run mode": RUN_MODE})

## Review prediction-only evidence

In [ ]:
cohort_root = OUTPUT_ROOT / "toysad8k-eligible-train-full-pool-4107-v1"
safe_root = cohort_root / "safe"
if safe_root.is_dir():
    display(pd.read_csv(safe_root / "view_contract_summary.csv"))
    display(pd.read_csv(safe_root / "review_signal_summary.csv"))
    display(NotebookImage(filename=str(safe_root / "multiview_completion.png")))
    panels = sorted((cohort_root / "review-only/figures").glob("*.png"))
    for path in panels[:12]:
        display(NotebookImage(filename=str(path)))
else:
    progress = json.loads((OUTPUT_ROOT / "tracking-safe/run_record.json").read_text(encoding="utf-8"))
    display(progress)

## Prediction publication boundary

Incremental output is a private recovery resource, not a scientific result. After full coverage, publish the cohort `model-safe`, `safe`, runtime and review-only evidence as the Notebook 11A output. Notebook 11B receives the locked predictions and the separately published evaluator resource. No Notebook 11A output can replace or revise the primary and official findings.